In [1]:
from autogen_agentchat.agents import AssistantAgent,SocietyOfMindAgent,UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_agentchat.conditions import TextMentionTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
model_client=OpenAIChatCompletionClient(model='gpt-4o')

In [8]:
async def main(task : str)-> None:
    agent_1=AssistantAgent(
        name="AssistantAgent_1",
        model_client=model_client,
        description="An Agent who writes short story on any subject.",
        system_message="""You are writer, you writes short story very well. 
        Write a short story within 50 words on the topic which user asked."""
    )
    agent_2=AssistantAgent(
        name="AssistantAgent_2",
        model_client=model_client,
        description=" An editor agent",
        system_message="""You are an editor, provide critical feedback within 30 words. 
        Respond with 'APPROVE' if the text addresses all feedbacks."""
    )
    user_proxy_agent=UserProxyAgent(
        name='user',
        description="An agent working as an user",
        input_func=input
    )
    inner_termination=TextMentionTermination("APPROVE")
    inner_team=RoundRobinGroupChat(participants=[agent_1,agent_2,user_proxy_agent],termination_condition=inner_termination)

    society_of_mind_agent=SocietyOfMindAgent(name="Society_of_mind",
                                             team=inner_team,model_client=model_client,
                                             response_prompt="""Output a standalone response to the 
                                             original request, without mentioning any of the 
                                             intermediate discussion response the final 
                                             response within 50 words""")
    
    agent_3=AssistantAgent(
        name="AssistantAgent_3",
        model_client=model_client,
        description="Language translator agent",
        system_message="""You are a translator who translate input text into 
        bengali language within 50 words. After completing your task raise 'TERMINATE' to exit.""")
    
    termination=TextMentionTermination('TERMINATE')
    team=RoundRobinGroupChat(participants=[society_of_mind_agent,agent_3],
                             max_turns=2,
                             termination_condition=termination)
    
    stream=team.run_stream(task=task)

    await Console(stream=stream)

    

In [9]:
await main(task="Write a story with suspances.")

---------- TextMessage (user) ----------
Write a story with suspances.
---------- TextMessage (AssistantAgent_1) ----------
Under the dim streetlight, Claire found an old key on her doorstep. Curious, she discovered it fit an abandoned mansion's lock nearby. Inside, whispers echoed through the empty halls. As she turned to leave, the door slammed shut. A voice murmured, "We've been waiting..." and the darkness consumed her.
---------- TextMessage (AssistantAgent_2) ----------
This story lacks adequate suspense buildup, character development, and contextual detail. Expand on Claire's emotions, introduce mysterious elements gradually, and establish a stronger backstory for greater intrigue.
---------- TextMessage (user) ----------
make the story more funny
---------- TextMessage (AssistantAgent_1) ----------
Under the dim streetlight, Claire found an oversized, cartoonish key on her doorstep with a note that said, "Open if you dare!" Intrigued, she discovered it fit an equally oversized 